In [ ]:
                                                                # CapstoneAzure

In [ ]:
                                        # Practical Assessment: Intelligent Document Processing + MLOps + AI Search

In [ ]:
                                                         # Task 1 - Extract Data from Documents

In [85]:
import requests
import time

In [86]:
import os
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("key")
endpoint="https://part1.cognitiveservices.azure.com/"

In [87]:
url = f"{endpoint}formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

In [88]:
headers = {
    "Ocp-Apim-Subscription-Key": key,
    "Content-Type": "application/octet-stream"
}

In [89]:
with open("image.png", "rb") as f:
    response = requests.post(url, headers=headers, data=f)

In [90]:
print("Status:", response.status_code)

Status: 202


In [95]:
if response.status_code == 202:
    operation_url = response.headers["Operation-Location"]

    while True:
        result = requests.get(operation_url, headers={
            "Ocp-Apim-Subscription-Key": key
        })

        result_json = result.json()

        if result_json["status"] == "succeeded":
            print("Extraction Completed")
            break
        elif result_json["status"] == "failed":
            print("Failed")
            break

        time.sleep(2)
else:
    print("Error:", response.text)

Extraction Completed


In [94]:
text = result_json["analyzeResult"]["content"]
print("\n--- EXTRACTED FIELDS ---\n")
lines = text.split("\n")

for i, line in enumerate(lines):
    if "Full Name" in line:
        print("Name:", lines[i+1])
    elif "Policy Number" in line:
        print("Policy Number:", lines[i+1])
    elif "Claim Amount" in line:
        print("Claim Amount:", lines[i+1].replace("₹", "").strip())
    elif "Date of Incident" in line:
        print("Date:", lines[i+1])


--- EXTRACTED FIELDS ---

Policy Number: P12345
Name: Rahul Sharma
Date: 05-02-2024
Claim Amount: 70,000


In [ ]:
                                                      # Task 2 - Create Searchable Index

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

endpoint = "https://search-himani.search.windows.net"
api_key = os.getenv("key")

index_name = "search-1776495273244"

search_url = f"{endpoint}/indexes/{index_name}/docs/search?api-version=2021-04-30-Preview"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}
query = {
    "search": "fraud",
    "filter": "Location eq 'Delhi'"
}
response = requests.post(search_url, headers=headers, json=query)

print(response.json())

In [ ]:
                                                      # TASK 4 - Deploy Model as Endpoint

In [25]:
import requests
import json
import os
from dotenv import load_dotenv

In [26]:
load_dotenv()
key = os.getenv("key")

# 🔹 Endpoint
url = "http://8aeb2c89-a50d-4944-bb8d-207c8d5d1808.southeastasia.azurecontainer.io/score"

In [27]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}"
}

In [28]:
customer_id = input("Enter Customer ID: ")
claim_amount = int(input("Enter Claim Amount: "))
claim_type = input("Enter Claim Type (Medical/Auto/etc): ")
location = input("Enter Location: ")
previous_claims = int(input("Enter Previous Claims: "))

Enter Customer ID:  C002
Enter Claim Amount:  150000
Enter Claim Type (Medical/Auto/etc):  Auto
Enter Location:  Mumbai
Enter Previous Claims:  5


In [29]:
data = {
    "Inputs": {
        "input1": [
            {
                "Customer_ID": customer_id,
                "Claim_Amount": claim_amount,
                "Claim_Type": claim_type,
                "Location": location,
                "Previous_Claims": previous_claims
            }
        ]
    },
    "GlobalParameters": {}
}

In [30]:
response = requests.post(url, headers=headers, json=data)

In [31]:
result = response.json()
output = result["Results"]["WebServiceOutput0"][0]

In [32]:
final_output = {
    "Customer_ID": output["Customer_ID"],
    "Claim_Amount": output["Claim_Amount"],
    "Claim_Type": output["Claim_Type"],
    "Location": output["Location"],
    "Previous_Claims": output["Previous_Claims"],
    "Fraud": "Yes" if output["Scored Labels"] else "No",
    "Confidence": round(output["Scored Probabilities"], 2)
}

In [34]:
print(json.dumps(final_output, indent=2))

{
  "Customer_ID": "C002",
  "Claim_Amount": 150000,
  "Claim_Type": "Auto",
  "Location": "Mumbai",
  "Previous_Claims": 5,
  "Fraud": "Yes",
  "Confidence": 0.69
}
